In [8]:
import pandas as pd

panel = pd.read_csv("../data/processed/panel_publico.csv")
eventos = pd.read_csv("../data/raw/eventos_medellin.csv")

In [9]:
eventos["fecha_inicio"] = pd.to_datetime(eventos["fecha_inicio"])
eventos["fecha_fin"] = pd.to_datetime(eventos["fecha_fin"])
eventos["fecha_fin"] = eventos["fecha_fin"].fillna(eventos["fecha_inicio"])  # eventos de un día
eventos["internacional"] = eventos["internacional"].str.strip().str.lower() == "si"

In [10]:
tipos_grandes = ["festival", "feria", "feria_comercial", "temporada"]
eventos["evento_grande"] = (
    (eventos["aforo_estimado"] >= 20000)
    | eventos["internacional"]
    | eventos["tipo"].isin(tipos_grandes)
    | eventos["escenario"].str.contains("Atanasio", na=False)
)
eventos["es_concierto"] = eventos["tipo"] == "concierto"
eventos["es_festival"] = eventos["tipo"].isin(tipos_grandes)

In [11]:
eventos["mes"] = eventos.apply(
    lambda fila: list(pd.period_range(fila["fecha_inicio"], fila["fecha_fin"], freq="M").astype(str)),
    axis=1,
)
eventos_largo = eventos.explode("mes")

In [12]:
eventos_mes = eventos_largo.groupby("mes").agg(
    n_eventos=("nombre", "count"),
    n_eventos_grandes=("evento_grande", "sum"),
    n_conciertos=("es_concierto", "sum"),
    n_festivales=("es_festival", "sum"),
    n_internacionales=("internacional", "sum"),
    aforo_total=("aforo_estimado", "sum"),
).reset_index()

In [13]:
final = panel.merge(eventos_mes, on="mes", how="left", validate="one_to_one")

columnas_eventos = ["n_eventos", "n_eventos_grandes", "n_conciertos",
                    "n_festivales", "n_internacionales", "aforo_total"]
final[columnas_eventos] = final[columnas_eventos].fillna(0).astype(int)

In [14]:
final["hay_evento_grande"] = (final["n_eventos_grandes"] > 0).astype(int)
final["covid"] = final["mes"].between("2020-03", "2021-12").astype(int)
final["temporada_alta"] = final["mes_num"].isin([1, 6, 7, 8, 12]).astype(int)

In [17]:
print("Filas:", final.shape[0], "| columnas:", final.shape[1])
print("Meses duplicados:", final["mes"].duplicated().sum())
print("Faltantes:", final.isna().sum().sum())
print("Meses de eventos que quedaron fuera:", sorted(set(eventos_mes["mes"]) - set(panel["mes"])))
final.to_csv("../data/processed/panel_final.csv", index=False)

Filas: 84 | columnas: 21
Meses duplicados: 0
Faltantes: 0
Meses de eventos que quedaron fuera: ['2026-01']
